# ElectInfo scales the ranking to Spark — call shape

## What this shows
How to move the `engines/01` top-committee ranking to a Spark cluster without rewriting the analysis layer. Spark sessions depend on a real cluster (or a properly-configured local JVM), so the Spark cells below document the call shape; they deliberately don't boot Spark inside the notebook — run them against your cluster directly, or within `spark-submit` / Databricks / EMR / standalone.

## Why it matters
When the Silver-layer FEC dataset grows past a single node's memory (40M+ rows), pandas and DuckDB stop being viable. Spark distributes the same operation. Sedona is the geospatial extension — ElectInfo uses it for precinct-scale point-in-polygon joins.

## Prereqs
- A Spark cluster (local or remote) with `pyspark` installed.
- For Sedona: `pip install apache-sedona` matching your Spark version, plus the Sedona JAR on the classpath.

## Next
- `engines/03_databricks_geo.ipynb` — Azure Databricks variant (no Sedona; uses the GeoPandas bridge pattern).
- `spatial/05_multi_source_joins.ipynb` — the kind of join Spark + Sedona targets at scale.


## 1. Boot a SparkSession

In a notebook that has a live cluster attached (Databricks, EMR, `pyspark --master ...`), this just works:

```python
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .appName('electinfo-fec-ranking')
        .master('yarn')            # or 'local[*]' for dev
        .config('spark.sql.warehouse.dir', 'hdfs:///user/electinfo/warehouse')
        .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
```


## 2. Same ranking, Spark backend

`SparkEngine` wraps a PySpark DataFrame with the same `groupby_agg` / `filter` / `join` surface as `PandasEngine` and `DuckDBEngine`. Only the constructor and the dataframe type change — the analysis code above it is identical to `engines/01`.

```python
from siege_utilities.engines.dataframe_engine import get_engine, Engine

spark_engine = get_engine(Engine.SPARK, spark=spark)
sdf = spark_engine.read_parquet('s3://electinfo-silver/fec/committees/')

ranked = (
    spark_engine
        .groupby_agg(sdf, group_cols=['committee_id'], agg_dict={'receipts': 'sum'})
        .withColumnRenamed('sum(receipts)', 'total_receipts')
        .orderBy('total_receipts', ascending=False)
        .limit(20)
        .toPandas()
)
```


## 3. Sedona geospatial join

Distributed point-in-polygon joins — precincts as polygons, donors/voters as lat/lon points — scale past anything a single geopandas process can do. This is ElectInfo's production pattern for precinct-level voter / donor aggregation.

```python
from sedona.spark import SedonaContext

config = (
    SedonaContext.builder()
        .config('spark.sql.extensions', 'org.apache.sedona.sql.SedonaSqlExtensions')
        .config('spark.kryo.registrator', 'org.apache.sedona.core.serde.SedonaKryoRegistrator')
        .getOrCreate()
)
sedona = SedonaContext.create(config)

joined = sedona.sql("""
    SELECT d.donor_id, p.precinct_id, SUM(d.amount) AS donor_total
    FROM donors d, precincts p
    WHERE ST_Contains(p.geom, ST_Point(d.lon, d.lat))
    GROUP BY d.donor_id, p.precinct_id
""")
```


## 4. Why this notebook doesn't boot Spark inline

Local Spark in a Jupyter kernel is fragile — JVM startup, port binding, serializer registration, and Py4J gateway lifetime all have to line up. In a cluster-attached environment (Databricks, EMR, `spark-submit --py-files`), the cells above execute as-is. In a laptop Jupyter kernel with a stock JVM, `spark.createDataFrame` often kills the executor on first shuffle. The right answer is "run these against your real cluster" — not "try to fake Spark in a notebook."

On Azure Databricks, where Sedona is not installed by default and adding it is non-trivial, the `dataframe_bridge` pattern in `engines/03_databricks_geo` is the supported workaround.

In [1]:
# Smoke-test: confirm siege_utilities.engines.SparkEngine is importable in this environment
# even if we don't exercise it against a live Spark session.
from siege_utilities.engines.dataframe_engine import Engine, SparkEngine
print('SparkEngine available      :', SparkEngine.__name__)
print('Engine enum values          :', [e.value for e in Engine])


SparkEngine available      : SparkEngine
Engine enum values          : ['pandas', 'duckdb', 'spark', 'postgis']


## Related

- **Source**: `siege_utilities/engines/dataframe_engine.py` (`SparkEngine`), `siege_utilities/distributed/` for vendor-agnostic Spark helpers.
- **Tests**: `tests/test_dataframe_engine_spatial.py`, `tests/test_distributed_*.py`.
- **Next**: `engines/03_databricks_geo.ipynb` — same patterns on Azure Databricks where Sedona isn't installable.
